# Question: Build a DataLoader in PyTorch

In this notebook, we build two PyTorch data pipelines:

1. A map-style dataset using `Dataset`.
2. A streaming dataset using `IterableDataset`.

We will also explain how `chunksize` (file reading) differs from `batch_size` (model input batching).

In [36]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, IterableDataset, get_worker_info

# Keep data generation reproducible for teaching.
np.random.seed(42)

## 1) Create a toy CSV dataset

We generate a simple linear relation $y = 2x + 1$ and save it as `data.csv`.

This file will be used by both dataset implementations below.

In [37]:
# Create a small CSV file for demonstration.
path = "data.csv"

x = np.random.rand(100, 1)
y = 2 * x + 1
data = np.hstack((x, y))
df = pd.DataFrame(data, columns=["x", "y"])

df.to_csv(path, index=False)
print(f"Saved {len(df)} rows to {path}")

Saved 100 rows to data.csv


## 2) Build a map-style dataset (`Dataset`)

A map-style dataset supports random access by index (`__getitem__`) and length (`__len__`).

This is the most common pattern when data can fit in memory.

In [38]:
class SimpleDataset(Dataset):
    def __init__(self, csv_path: str) -> None:
        data = pd.read_csv(csv_path)
        self.x = torch.tensor(data["x"].to_numpy(), dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(data["y"].to_numpy(), dtype=torch.float32).unsqueeze(1)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]

## 3) Create a DataLoader for the map-style dataset

`DataLoader` groups samples into mini-batches.

Here, `batch_size=16` means each iteration returns up to 16 samples.

In [39]:
dataset = SimpleDataset(path)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

for batch_x, batch_y in dataloader:
    print("Map-style batch:", batch_x.shape, batch_y.shape)
    break

Map-style batch: torch.Size([16, 1]) torch.Size([16, 1])


## 4) Build a streaming dataset (`IterableDataset`)

Use `IterableDataset` when data is too large to fully load into memory.

We read the CSV in chunks and yield samples one-by-one to the DataLoader.

In [41]:
class LargeCSVDataset(IterableDataset):
    def __init__(self, file_path: str, chunksize: int = 10) -> None:
        self.file_path = file_path
        self.chunksize = chunksize

    def __iter__(self):
        worker_info = get_worker_info()

        if worker_info is None:
            start = 0
            step = 1
        else:
            # Split chunk indices across workers.
            start = worker_info.id
            step = worker_info.num_workers

        reader = pd.read_csv(self.file_path, chunksize=self.chunksize)

        for chunk_idx, chunk in enumerate(reader):
            if chunk_idx % step != start:
                continue

            x_vals = chunk["x"].to_numpy(dtype="float32")
            y_vals = chunk["y"].to_numpy(dtype="float32")

            for x_i, y_i in zip(x_vals, y_vals):
                # Return 1D tensors so batched output is shape [batch_size, 1].
                yield torch.tensor([x_i]), torch.tensor([y_i])


stream_dataset = LargeCSVDataset(path, chunksize=10)
stream_loader = DataLoader(
    stream_dataset,
    batch_size=64,
    num_workers=0,  # Safer in notebooks on Windows.
    pin_memory=torch.cuda.is_available(),
)

for batch_x, batch_y in stream_loader:
    print("Iterable batch:", batch_x.shape, batch_y.shape)
    break

Iterable batch: torch.Size([64, 1]) torch.Size([64, 1])


## 5) What does `chunksize` control?

`chunksize` and `batch_size` are different:

- `chunksize`: how many rows Pandas reads from disk at one time.
- `batch_size`: how many samples DataLoader groups into one model batch.

Example in this notebook:

- `chunksize=10`: CSV is read in 10-row pieces.
- `batch_size=64`: DataLoader keeps collecting yielded samples across many chunks until it has 64 samples, then returns one batch.

So seeing output shape 64 is expected even when chunksize is 10.